In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D1 — CEDEFOP Labour Skills Shortage Dataset

# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

import json
import hashlib
import shutil
from pathlib import Path

import openpyxl
from google.colab import files

In [ ]:
# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DOCUMENT_ID = "D1"
DOCUMENT_NAME = "CEDEFOP Labour Skills Shortage Dataset"

BRANCH_ID = "A"
BRANCH_NAME = "Direct Ingestion"

EXPECTED_SOURCE_FORMAT = ".xlsx"

EXPECTED_SHEETS = [
    "EU27",
    "IT",
    "NL",
    "PT"
]

EXPECTED_RECORD_COUNT = 156

EXPECTED_RECORD_FIELDS = [
    "Geographic Area",
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Index",
    "LSI (Comp.)",
    "LSI1",
    "LSI2",
    "LSI3"
]

NUMERIC_FIELDS = [
    "Labour Shortage Index",
    "LSI1",
    "LSI2",
    "LSI3"
]

OUTPUT_DIR = Path("outputs_D1_branch_A")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Experiment configured.")

In [ ]:
# ------------------------------------------------------------
# 2. Source document upload
# ------------------------------------------------------------

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError(
        "Upload exactly one source document for D1."
    )

FILE_PATH = Path(next(iter(uploaded)))

print(f"Loaded source file: {FILE_PATH}")

In [ ]:
# ------------------------------------------------------------
# 3. Source format verification
# ------------------------------------------------------------

if FILE_PATH.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_FORMAT} source file, "
        f"received {FILE_PATH.suffix}"
    )

print("Source format verified.")

In [ ]:
# ------------------------------------------------------------
# 4. Source SHA-256
# ------------------------------------------------------------

def calculate_sha256(file_path, chunk_size=8192):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


SOURCE_SHA256 = calculate_sha256(FILE_PATH)

print("Source SHA-256:")
print(SOURCE_SHA256)

In [ ]:
# ------------------------------------------------------------
# 5. Workbook structure verification
# ------------------------------------------------------------

workbook = openpyxl.load_workbook(
    FILE_PATH,
    read_only=True,
    data_only=True
)

observed_sheets = workbook.sheetnames

print("Observed worksheets:")
print(observed_sheets)

if observed_sheets != EXPECTED_SHEETS:
    raise ValueError(
        "Unexpected D1 worksheet structure.\n"
        f"Expected: {EXPECTED_SHEETS}\n"
        f"Observed: {observed_sheets}"
    )

print("Workbook worksheet structure verified.")

In [ ]:
# ------------------------------------------------------------
# 6. Branch A representation
# ------------------------------------------------------------

branch_representation = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "representation_type": "Original source document",
    "input_file": FILE_PATH.name,
    "input_format": FILE_PATH.suffix.lower(),
    "structural_conversion_applied": False,
    "normalisation_applied": False,
    "ocr_applied": False,
    "derived_representation_used_as_model_input": False,
    "model_input_description":
        "The original XLSX workbook is submitted directly to the LLM."
}

print(
    json.dumps(
        branch_representation,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ------------------------------------------------------------
# 7. Extraction schema
# ------------------------------------------------------------

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "records": [
        {
            "Geographic Area": None,
            "Main Occupation Group": None,
            "Occupation Group (2 digit)": None,
            "Labour Shortage Index": None,
            "LSI (Comp.)": None,
            "LSI1": None,
            "LSI2": None,
            "LSI3": None
        }
    ]
}

print(
    json.dumps(
        EXTRACTION_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ------------------------------------------------------------
# 8. Fixed extraction task
# ------------------------------------------------------------

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract the labour shortage information from the attached original
CEDEFOP Labour Skills Shortage Index Excel workbook.

Process every worksheet in the workbook.

Return one record for each occupation-group observation.

For each record, extract:
- Geographic Area
- Main Occupation Group
- Occupation Group (2 digit)
- Labour Shortage Index
- LSI (Comp.)
- LSI1
- LSI2
- LSI3

Extraction rules:
- Extract only information explicitly supported by the workbook.
- Use the worksheet name as the Geographic Area.
- Preserve the association between each geographic area, main occupation
  group, two-digit occupation group and corresponding LSI values.
- Preserve LSI (Comp.) exactly as represented in the workbook.
- Return LSI1, LSI2 and LSI3 as numerical values.
- Return Labour Shortage Index as a numerical value.
- Use null only when a requested value is not available.
- Do not infer, calculate, reconstruct or invent missing values.
- Do not omit repeated Main Occupation Group values from individual records.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
"""

In [ ]:
# ------------------------------------------------------------
# 9. Extraction prompt
# ------------------------------------------------------------

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(EXTRACTION_SCHEMA, indent=2, ensure_ascii=False)}

The original Excel workbook is attached as the extraction source.

Return only the JSON object.
""".strip()

PROMPT_PATH = (
    OUTPUT_DIR /
    "D1_branch_A_prompt.txt"
)

PROMPT_PATH.write_text(
    FULL_PROMPT,
    encoding="utf-8"
)

print(FULL_PROMPT)
print()
print(f"Prompt saved to: {PROMPT_PATH}")

In [ ]:
# ------------------------------------------------------------
# 10. Experiment metadata
# ------------------------------------------------------------

experiment_metadata = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,

    "branch": BRANCH_ID,
    "branch_name": BRANCH_NAME,

    "source_file": FILE_PATH.name,
    "source_format": FILE_PATH.suffix.lower(),
    "source_sha256": SOURCE_SHA256,

    "source_structure": {
        "expected_sheets": EXPECTED_SHEETS,
        "observed_sheets": observed_sheets,
        "sheet_structure_verified":
            observed_sheets == EXPECTED_SHEETS
    },

    "input_representation":
        "Original XLSX workbook",

    "structural_conversion_applied": False,
    "normalisation_applied": False,
    "ocr_applied": False,

    "expected_extraction_scope": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "expected_fields": EXPECTED_RECORD_FIELDS
    },

    "prompt_file": PROMPT_PATH.name,

    "execution_environment":
        "Independent ChatGPT conversation",

    "expected_output_format":
        "JSON",

    "notes":
        "The original workbook is submitted directly to the LLM. "
        "Source inspection performed in this notebook is diagnostic only "
        "and does not modify the model input."
}

METADATA_PATH = (
    OUTPUT_DIR /
    "D1_branch_A_experiment_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        experiment_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        experiment_metadata,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ------------------------------------------------------------
# 11. Representation metadata
# ------------------------------------------------------------

REPRESENTATION_PATH = (
    OUTPUT_DIR /
    "D1_branch_A_representation.json"
)

with open(
    REPRESENTATION_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        branch_representation,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Representation metadata saved to: {REPRESENTATION_PATH}")
files.download(PROMPT_PATH)

## Independent Branch A extraction

1. Open a new independent ChatGPT conversation.
2. Upload the original D1 XLSX workbook.
3. Submit the exact contents of `D1_branch_A_prompt.txt`.
4. Save the complete response exactly as returned.

In [ ]:
# ------------------------------------------------------------
# 12. Raw response upload
# ------------------------------------------------------------

uploaded_output = files.upload()

if len(uploaded_output) != 1:
    raise ValueError(
        "Upload exactly one file containing the complete "
        "raw D1 Branch A LLM response."
    )

RAW_OUTPUT_PATH = Path(
    next(iter(uploaded_output))
)

print(f"Uploaded model response: {RAW_OUTPUT_PATH}")

In [ ]:
# ------------------------------------------------------------
# 13. Raw-response preservation
# ------------------------------------------------------------

RAW_RESPONSE_PATH = (
    OUTPUT_DIR /
    "D1_branch_A_raw_response.txt"
)

raw_response_text = RAW_OUTPUT_PATH.read_text(
    encoding="utf-8"
)

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)

print(
    "Raw LLM response preserved exactly as supplied "
    "before parsing or technical diagnostics."
)

In [ ]:
# ------------------------------------------------------------
# 14. Raw-response parsing
# ------------------------------------------------------------

json_valid = False
json_error = None
raw_extraction = None

try:
    raw_extraction = json.loads(
        raw_response_text
    )
    json_valid = True

except json.JSONDecodeError as error:
    json_error = str(error)

print(f"Valid JSON: {json_valid}")

if json_error:
    print("JSON parsing error:")
    print(json_error)

In [ ]:
# ------------------------------------------------------------
# 15. Top-level structure diagnostics
# ------------------------------------------------------------

top_level_checks = {
    "output_is_json_object": False,
    "document_id_present": False,
    "document_id_correct": False,
    "branch_present": False,
    "branch_correct": False,
    "records_present": False,
    "records_is_list": False
}

if json_valid and isinstance(
    raw_extraction,
    dict
):
    top_level_checks[
        "output_is_json_object"
    ] = True

    top_level_checks[
        "document_id_present"
    ] = "document_id" in raw_extraction

    top_level_checks[
        "document_id_correct"
    ] = (
        raw_extraction.get("document_id")
        == DOCUMENT_ID
    )

    top_level_checks[
        "branch_present"
    ] = "branch" in raw_extraction

    top_level_checks[
        "branch_correct"
    ] = (
        raw_extraction.get("branch")
        == BRANCH_ID
    )

    top_level_checks[
        "records_present"
    ] = "records" in raw_extraction

    top_level_checks[
        "records_is_list"
    ] = isinstance(
        raw_extraction.get("records"),
        list
    )

print(
    json.dumps(
        top_level_checks,
        indent=2
    )
)

In [ ]:
# ------------------------------------------------------------
# 16. Record-count diagnostics
# ------------------------------------------------------------

observed_record_count = None
record_count_matches = False

if (
    json_valid
    and isinstance(raw_extraction, dict)
    and isinstance(
        raw_extraction.get("records"),
        list
    )
):
    observed_record_count = len(
        raw_extraction["records"]
    )

    record_count_matches = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )

record_count_check = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        observed_record_count,
    "record_count_matches":
        record_count_matches
}

print(
    json.dumps(
        record_count_check,
        indent=2
    )
)

In [ ]:
# ------------------------------------------------------------
# 17. Record-structure diagnostics
# ------------------------------------------------------------

record_structure_issues = []

if (
    json_valid
    and isinstance(raw_extraction, dict)
    and isinstance(
        raw_extraction.get("records"),
        list
    )
):

    expected_field_set = set(
        EXPECTED_RECORD_FIELDS
    )

    for index, record in enumerate(
        raw_extraction["records"]
    ):

        if not isinstance(record, dict):
            record_structure_issues.append({
                "record_index": index,
                "issue":
                    "Record is not a JSON object"
            })
            continue

        actual_fields = set(
            record.keys()
        )

        missing_fields = sorted(
            expected_field_set -
            actual_fields
        )

        additional_fields = sorted(
            actual_fields -
            expected_field_set
        )

        if (
            missing_fields
            or additional_fields
        ):
            record_structure_issues.append({
                "record_index": index,
                "missing_fields":
                    missing_fields,
                "additional_fields":
                    additional_fields
            })

print(
    f"Records with structural issues: "
    f"{len(record_structure_issues)}"
)

In [ ]:
# ------------------------------------------------------------
# 18. Field-type diagnostics
# ------------------------------------------------------------

field_type_issues = []

if (
    json_valid
    and isinstance(raw_extraction, dict)
    and isinstance(
        raw_extraction.get("records"),
        list
    )
):

    for index, record in enumerate(
        raw_extraction["records"]
    ):

        if not isinstance(record, dict):
            continue

        for field in NUMERIC_FIELDS:

            if field not in record:
                continue

            value = record[field]

            if value is None:
                continue

            if not isinstance(
                value,
                (int, float)
            ):
                field_type_issues.append({
                    "record_index": index,
                    "field": field,
                    "observed_type":
                        type(value).__name__,
                    "observed_value":
                        value
                })

print(
    f"Numeric field type issues: "
    f"{len(field_type_issues)}"
)

In [ ]:
# ------------------------------------------------------------
# 19. Technical diagnostic summary
# ------------------------------------------------------------

structurally_evaluable = all([
    json_valid,
    top_level_checks["output_is_json_object"],
    top_level_checks["document_id_correct"],
    top_level_checks["branch_correct"],
    top_level_checks["records_present"],
    top_level_checks["records_is_list"],
    len(record_structure_issues) == 0,
    len(field_type_issues) == 0
])

structure_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,

    "json_valid":
        json_valid,

    "json_error":
        json_error,

    "top_level_checks":
        top_level_checks,

    "structurally_evaluable":
        structurally_evaluable,

    "record_count_check":
        record_count_check,

    "records_with_structure_issues":
        len(record_structure_issues),

    "record_structure_issues":
        record_structure_issues,

    "numeric_field_type_issues":
        len(field_type_issues),

    "field_type_issues":
        field_type_issues
}

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR /
    "D1_branch_A_technical_diagnostics.json"
)

with open(
    TECHNICAL_DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        structure_check,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        structure_check,
        indent=2,
        ensure_ascii=False
    )
)


In [ ]:
# ------------------------------------------------------------
# 20. Parsed extraction
# ------------------------------------------------------------

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR /
    "D1_branch_A_parsed_extraction.json"
)

if json_valid:

    with open(
        PARSED_EXTRACTION_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            raw_extraction,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"Parsed extraction saved to: "
        f"{PARSED_EXTRACTION_PATH}"
    )

else:
    print(
        "Parsed extraction was not created "
        "because the raw response is not valid JSON."
    )

In [ ]:
# ------------------------------------------------------------
# 21. Experiment summary
# ------------------------------------------------------------

experiment_summary = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,

    "source_file":
        FILE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        observed_sheets == EXPECTED_SHEETS,

    "input_representation":
        "Original XLSX workbook",

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "ocr_applied":
        False,

    "json_valid":
        json_valid,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "structurally_evaluable":
        structurally_evaluable,

    "record_count_matches":
        record_count_matches,

    "records_with_structure_issues":
        len(record_structure_issues),

    "field_type_issues":
        len(field_type_issues),

    "raw_response_preserved":
        True,

    "content_validation_performed":
        False,

    "parsed_extraction_created":
        json_valid
}

SUMMARY_PATH = (
    OUTPUT_DIR /
    "D1_branch_A_experiment_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        experiment_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        experiment_summary,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ------------------------------------------------------------
# 22. Final artefact inventory
# ------------------------------------------------------------

outputs_created = [
    PROMPT_PATH,
    METADATA_PATH,
    REPRESENTATION_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    SUMMARY_PATH
]

if json_valid:
    outputs_created.append(
        PARSED_EXTRACTION_PATH
    )

print("Branch A outputs created:")

for output in outputs_created:
    print(f"- {output}")

In [ ]:
# ------------------------------------------------------------
# 23. Download experiment artefacts
# ------------------------------------------------------------

for output in outputs_created:
    files.download(output)